# Zadanie 3
Wykorzystaj przykłady z notatnika w SQL Windowed Aggregate Functions (cmd 11) i przepisz funkcje używając Spark API


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import col
from datetime import datetime

schema = StructType([
    StructField("AccountId", IntegerType(), True),
    StructField("TranDate", DateType(), True),
    StructField("TranAmt", DoubleType(), True)
])

data = [
    (1, datetime(2025, 3, 1), 200.0),
    (1, datetime(2025, 3, 2), 150.0),
    (1, datetime(2025, 3, 3), 100.0),
    (1, datetime(2025, 3, 4), 50.0),
    (1, datetime(2025, 3, 5), 300.0),
    (2, datetime(2025, 3, 1), 600.0),
    (2, datetime(2025, 3, 2), 400.0),
    (2, datetime(2025, 3, 3), 500.0),
    (2, datetime(2025, 3, 4), 800.0),
    (2, datetime(2025, 3, 5), 700.0),
]

df = spark.createDataFrame(data, schema=schema)
display(df)

AccountId,TranDate,TranAmt
1,2025-03-01,200.0
1,2025-03-02,150.0
1,2025-03-03,100.0
1,2025-03-04,50.0
1,2025-03-05,300.0
2,2025-03-01,600.0
2,2025-03-02,400.0
2,2025-03-03,500.0
2,2025-03-04,800.0
2,2025-03-05,700.0


In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as FUN
window_spec = Window.partitionBy("AccountId").orderBy("TranDate")
df.withColumn("RunTotalAmt", FUN.sum("TranAmt").over(window_spec)).orderBy(["AccountId","TranDate"]).show()



+---------+----------+-------+-----------+
|AccountId|  TranDate|TranAmt|RunTotalAmt|
+---------+----------+-------+-----------+
|        1|2025-03-01|  200.0|      200.0|
|        1|2025-03-02|  150.0|      350.0|
|        1|2025-03-03|  100.0|      450.0|
|        1|2025-03-04|   50.0|      500.0|
|        1|2025-03-05|  300.0|      800.0|
|        2|2025-03-01|  600.0|      600.0|
|        2|2025-03-02|  400.0|     1000.0|
|        2|2025-03-03|  500.0|     1500.0|
|        2|2025-03-04|  800.0|     2300.0|
|        2|2025-03-05|  700.0|     3000.0|
+---------+----------+-------+-----------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, count, min, max, sum, row_number

# Zastosowanie funkcji okienkowych
df_transformed = df.withColumn("SlideAvg", avg("TranAmt").over(window_spec)) \
                   .withColumn("SlideQty", count("*").over(window_spec)) \
                   .withColumn("SlideMin", min("TranAmt").over(window_spec)) \
                   .withColumn("SlideMax", max("TranAmt").over(window_spec)) \
                   .withColumn("SlideTotal", sum("TranAmt").over(window_spec)) \
                   .withColumn("RN", row_number().over(Window.partitionBy("AccountId").orderBy("TranDate")))

# Wyświetlenie wyniku
df_transformed.orderBy("AccountId", "TranDate", "RN").show()

+---------+----------+-------+--------+--------+--------+--------+----------+---+
|AccountId|  TranDate|TranAmt|SlideAvg|SlideQty|SlideMin|SlideMax|SlideTotal| RN|
+---------+----------+-------+--------+--------+--------+--------+----------+---+
|        1|2025-03-01|  200.0|   200.0|       1|   200.0|   200.0|     200.0|  1|
|        1|2025-03-02|  150.0|   175.0|       2|   150.0|   200.0|     350.0|  2|
|        1|2025-03-03|  100.0|   150.0|       3|   100.0|   200.0|     450.0|  3|
|        1|2025-03-04|   50.0|   125.0|       4|    50.0|   200.0|     500.0|  4|
|        1|2025-03-05|  300.0|   160.0|       5|    50.0|   300.0|     800.0|  5|
|        2|2025-03-01|  600.0|   600.0|       1|   600.0|   600.0|     600.0|  1|
|        2|2025-03-02|  400.0|   500.0|       2|   400.0|   600.0|    1000.0|  2|
|        2|2025-03-03|  500.0|   500.0|       3|   400.0|   600.0|    1500.0|  3|
|        2|2025-03-04|  800.0|   575.0|       4|   400.0|   800.0|    2300.0|  4|
|        2|2025-

# Zadanie 2
Do tego notatnika dopisz użycie funkcji okienkowych LEAD, LAG, FIRST_VALUE, LAST_VALUE, ROW_NUMBER 


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, lag, first, last, row_number

# Definiujemy okno dla każdej grupy AccountId, uporządkowane według TranDate
window_spec = Window.partitionBy("AccountId").orderBy("TranDate")

df_windowed = df_transformed.withColumn("PrevTranAmt", lag("TranAmt", 1).over(window_spec))\
    .withColumn("NextTranAmt", lead("TranAmt", 1).over(window_spec))\
    .withColumn("FirstTranAmt", first("TranAmt").over(window_spec))\
    .withColumn("LastTranAmt", last("TranAmt").over(window_spec))\
    .withColumn("RN", row_number().over(window_spec))

df_windowed.show()

+---------+----------+-------+--------+--------+--------+--------+----------+---+-----------+-----------+------------+-----------+
|AccountId|  TranDate|TranAmt|SlideAvg|SlideQty|SlideMin|SlideMax|SlideTotal| RN|PrevTranAmt|NextTranAmt|FirstTranAmt|LastTranAmt|
+---------+----------+-------+--------+--------+--------+--------+----------+---+-----------+-----------+------------+-----------+
|        1|2025-03-01|  200.0|   200.0|       1|   200.0|   200.0|     200.0|  1|       null|      150.0|       200.0|      200.0|
|        1|2025-03-02|  150.0|   175.0|       2|   150.0|   200.0|     350.0|  2|      200.0|      100.0|       200.0|      150.0|
|        1|2025-03-03|  100.0|   150.0|       3|   100.0|   200.0|     450.0|  3|      150.0|       50.0|       200.0|      100.0|
|        1|2025-03-04|   50.0|   125.0|       4|    50.0|   200.0|     500.0|  4|      100.0|      300.0|       200.0|       50.0|
|        1|2025-03-05|  300.0|   160.0|       5|    50.0|   300.0|     800.0|  5|  